# Problem Statement
**Input:** Ảnh bài làm học sinh + ảnh đề mẫu (template) + file template_roi.json từ module 1 

**Output:** Ảnh bài làm học sinh đã được cut sát ô điền đáp án, index theo từng câu, từng hs.

## Methodology
1. Làm ROI mapping: một câu có bao nhiêu chỗ điền đáp án?
2. Align ảnh bài làm học sinh với template đề. Hàm align_images.
3. Áp các tọa độ ROI đã tìm trên template mẫu sang bài làm của học sinh
4. Thêm cơ chế fallback, phát hiện sớm nếu định thức warp lỗi, không cắt ảnh và chép lại vào file log.json.

## Discussion
Hầu hết các ảnh đều khớp tốt, trong đó có một số ít ảnh còn lỗi như sau:
- Tờ đề số 7: còn bị lệch một chút xuống dưới, khiến bị mất dòng đầu tiên trong bài làm của học sinh
- Tờ đề số 9: cũng bị lệch một chút xuống dưới, còn có vài ảnh bị warp hỏng, nguyên nhân có thể do các điểm neo mà thuật toán tìm được đều nằm gần nhau ở nửa trên template, khiến ma trận chuyển đổi làm nát ảnh

Ngoài ra vì RANSAC là thuật toán có tính ngẫu nhiên, có thể trong lúc lấy điểm neo lấy trúng nhiễu thì sẽ bị lỗi, cũng có khả năng lấy đúng điểm neo.

Câu hỏi thêm:
- Có nên tăng số lượng ROI lên để detect theo từng dòng bài làm? (thế thì phải assume học sinh viết theo thứ tự từ trên xuống dưới)

## Future Work
- Nhận thấy có một số học sinh làm bài vẫn sạch đẹp, chỉ là điền bị dôi ra vùng bên phải, nên có thể sẽ bổ sung file khác ROI dài hơn ở phía đó
- Thử điều chỉnh nhẹ hoặc thêm các dấu chấm lớn làm nút neo ở những tờ đề mà bị warp hỏng

# Import thư viện, tắt Warning

In [ ]:
import logging

# Ép bộ logger của hệ thống chỉ in ra LỖI (ERROR) làm sập chương trình, 
# phớt lờ tất cả các CẢNH BÁO (WARNING) lặt vặt.
logging.getLogger("root").setLevel(logging.ERROR)

In [ ]:
import cv2
import math
import json
import os
import re
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

In [ ]:
ROI_NAMING_MAP = [
    "Cau_01",
    "Cau_02",
    "Cau_03",  # Câu 3 có 2 vùng, nhưng vì ROI bắt thành 1 vùng nên việc xử lí sẽ để module 3
    "Cau_04",
    "Cau_05",
    "Cau_06",
    "Cau_07",
    "Cau_08_1", "Cau_08_2",  # Câu 8 có 2 vùng
    "Cau_09",
    "Cau_10",
    "Cau_11",
    "Cau_12",
    "Cau_13a", "Cau_13b", "Cau_13c",       # Câu 13 có 3 vùng
    "Cau_14a", "Cau_14b", "Cau_14c",       # Câu 14 có 3 vùng
    "Cau_15a",                             # Câu 15: Vùng 1
    "Cau_15b", # Câu 15b là bảng
    "Cau_15c"                              # Câu 15: Vùng 5 (Sơ đồ)
]

In [ ]:
PAGE_MAP = {
    "Trang_01": ["Cau_01", "Cau_02"],
    "Trang_02": ["Cau_03", "Cau_04", "Cau_05"],
    "Trang_03": ["Cau_06", "Cau_07"],
    "Trang_04": ["Cau_08_1", "Cau_08_2", "Cau_09"],
    "Trang_05": ["Cau_10", "Cau_11"],
    "Trang_06": ["Cau_12", "Cau_13a"],
    "Trang_07": ["Cau_13b", "Cau_13c", "Cau_14a"],
    "Trang_08": ["Cau_14b", "Cau_14c"],
    "Trang_09": ["Cau_15a", "Cau_15b", "Cau_15c"]
}

# Áp file ROI đã có lên bài làm của học sinh

In [ ]:
# --- [THÊM MỚI]: HÀM KIỂM TRA ĐỘ BIẾN DẠNG HOMOGRAPHY ---
def verify_homography(H, template_shape):
    if H is None: return False, "H_matrix is None"
    
    det = np.linalg.det(H[0:2, 0:2])
    if det <= 0.05 or det >= 15.0:
        return False, f"Định thức H bất thường ({det:.2f}), ảnh bị bóp méo quá mức."

    # =========================================================
    # [CẬP NHẬT]: CHẶN LỖI MA TRẬN ẢO GIÁC LÀM MÉO GIẤY THÀNH HÌNH THOI
    # =========================================================
    h, w = template_shape[:2]
    pts_template = np.float32([[0, 0], [0, h-1], [w-1, h-1], [w-1, 0]]).reshape(-1, 1, 2)
    pts_student = cv2.perspectiveTransform(pts_template, H).reshape(-1, 2)
    
    p0, p1, p2, p3 = pts_student[0], pts_student[1], pts_student[2], pts_student[3]
    v01 = p1 - p0 # Cạnh trái
    v03 = p3 - p0 # Cạnh trên
    
    def cosine_angle(vA, vB):
        norm_A, norm_B = np.linalg.norm(vA), np.linalg.norm(vB)
        if norm_A == 0 or norm_B == 0: return 1.0 
        return np.dot(vA, vB) / (norm_A * norm_B)

    cos_theta = cosine_angle(v01, v03)
    
    # Nới lỏng ngưỡng: Cho phép lệch đến ~11.5 độ (cos(78.5) ≈ 0.20).
    # Nếu lớn hơn 0.20 nghĩa là tờ giấy bị vặn xéo quá lố do ORB bắt nhầm điểm.
    if abs(cos_theta) > 0.20:
        angle_deg = np.degrees(np.arccos(np.clip(cos_theta, -1.0, 1.0)))
        return False, f"Lỗi Skew Toán học: Ma trận làm biến dạng giấy quá lố ({angle_deg:.1f}°)."

    return True, "Hợp lệ"

In [ ]:
def check_image_skew(aligned_img, skew_threshold=2.0):
    """
    Dò góc nghiêng bằng phương pháp Text Block Contours (Đóng khối chữ).
    Bất chấp nét đứt, chữ viết tay loằng ngoằng và bỏ qua các đường gạch chéo.
    """
    gray = cv2.cvtColor(aligned_img, cv2.COLOR_BGR2GRAY)
    
    # 1. Binarize (Chuyển đen trắng)
    # Dùng Adaptive Threshold để bóc tách chữ khỏi nền giấy (tránh nhiễu bóng râm)
    thresh = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 21, 10)
    
    # 2. Bôi nhòe theo chiều ngang (Kết dính chữ & nét đứt)
    # Kernel 40x3 sẽ quét ngang, biến các dòng chấm đứt/chữ viết thành 1 dải ruy-băng liền mạch
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (40, 3))
    dilated = cv2.dilate(thresh, kernel, iterations=1)
    
    # 3. Tìm các khối hình học (Contours)
    contours, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    angles = []
    
    for cnt in contours:
        # Lấy hình chữ nhật bao quanh khối chữ (Hỗ trợ bao quanh cả khối bị nghiêng)
        rect = cv2.minAreaRect(cnt)
        box = cv2.boxPoints(rect)
        
        # Tìm cạnh dài nhất của hình chữ nhật để làm chuẩn đo góc
        p0, p1, p2, p3 = box
        edges = [(p0, p1), (p1, p2), (p2, p3), (p3, p0)]
        longest_edge = max(edges, key=lambda e: (e[0][0]-e[1][0])**2 + (e[0][1]-e[1][1])**2)
        
        dx = longest_edge[1][0] - longest_edge[0][0]
        dy = longest_edge[1][1] - longest_edge[0][1]
        
        # Tính độ dài khối, bỏ qua các chấm nhiễu bé tí (< 100 pixels)
        length = np.sqrt(dx**2 + dy**2)
        if length < 100: continue 
            
        # Tính góc nghiêng
        angle = np.degrees(np.arctan2(dy, dx))
        
        # Đưa góc về chuẩn [-90, 90]
        if angle > 90: angle -= 180
        elif angle < -90: angle += 180
            
        # Chỉ xét các dải ruy-băng có xu hướng nằm ngang [-45, 45]
        # (Lệnh này sẽ TỰ ĐỘNG BỎ QUA đường gạch chéo đỏ khổng lồ của giáo viên vì nó nghiêng > 45 độ)
        if -45 < angle < 45:
            angles.append(angle)
            
    # 4. Kiểm định an toàn
    if len(angles) < 3:
        return True, 0.0 # Bỏ qua nếu trang giấy không có chữ
        
    # Lấy trung vị (Góc đại diện chính xác nhất của toàn bộ trang giấy)
    median_angle = np.median(angles)
    
    # 5. So sánh với ngưỡng an toàn
    if abs(median_angle) > skew_threshold:
        return False, abs(median_angle)
        
    return True, abs(median_angle)

In [ ]:
# ==========================================
# HÀM ALIGN: TRẢ VỀ ẢNH VÀ CHUẨN HÓA ERROR INFO
# ==========================================
def module_1_align_images(template_img, student_img, max_features=5000, match_percent=0.15):
    gray_temp = cv2.cvtColor(template_img, cv2.COLOR_BGR2GRAY)
    gray_stud = cv2.cvtColor(student_img, cv2.COLOR_BGR2GRAY)
    
    orb = cv2.ORB_create(max_features)
    keypoints_temp, descriptors_temp = orb.detectAndCompute(gray_temp, None)
    keypoints_stud, descriptors_stud = orb.detectAndCompute(gray_stud, None)
    
    # [LỖI LỚP 1]: KHÔNG THỂ WARP
    if descriptors_temp is None or descriptors_stud is None:
        return student_img, {"error_type": "FEATURE_ERROR", "reason": "Ảnh quá mờ hoặc trống."}

    matcher = cv2.DescriptorMatcher_create(cv2.DESCRIPTOR_MATCHER_BRUTEFORCE_HAMMING)
    matches = matcher.match(descriptors_stud, descriptors_temp)
    matches = sorted(matches, key=lambda x: x.distance)
    
    num_good_matches = int(len(matches) * match_percent)
    matches = matches[:num_good_matches]
    
    # [LỖI LỚP 2]: KHÔNG THỂ WARP
    if len(matches) < 10:
        return student_img, {"error_type": "MATCH_ERROR", "reason": f"Quá ít điểm khớp ({len(matches)})."}

    points_stud = np.zeros((len(matches), 2), dtype=np.float32)
    points_temp = np.zeros((len(matches), 2), dtype=np.float32)
    for i, match in enumerate(matches):
        points_stud[i, :] = keypoints_stud[match.queryIdx].pt
        points_temp[i, :] = keypoints_temp[match.trainIdx].pt
        
    h_matrix, mask = cv2.findHomography(points_stud, points_temp, cv2.RANSAC)
    
    # [LỖI LỚP 3]: KHÔNG THỂ WARP
    if h_matrix is None:
        return student_img, {"error_type": "HOMOGRAPHY_ERROR", "reason": "Không thể tính toán ma trận."}

    # ==========================================
    # [CẬP NHẬT]: ĐÃ CÓ MA TRẬN LÀ ÉP WARP LUÔN!
    # Dù ma trận có tốt hay xấu, cứ nắn ra xem hình thù thế nào đã
    # ==========================================
    h, w = template_img.shape[:2]
    aligned_img = cv2.warpPerspective(student_img, h_matrix, (w, h), borderValue=(255, 255, 255))

    # [LỖI LỚP 4]: TRẢ VỀ ẢNH WARP KÌ DỊ
    is_valid, reason = verify_homography(h_matrix, template_img.shape)
    if not is_valid:
        # Bắt được ma trận dị dạng -> Trả về ảnh nắn lỗi (aligned_img) để vẽ Box debug
        return aligned_img, {"error_type": "GEOMETRY_WARP_ERROR", "reason": reason}
    
    # [LỖI LỚP 5]: TRẢ VỀ ẢNH WARP BỊ NGHIÊNG DÒNG KẺ
    is_hough_safe, actual_skew = check_image_skew(aligned_img, skew_threshold=1.5)
    if not is_hough_safe:
        return aligned_img, {"error_type": "HOUGH_SKEW_ERROR", "reason": f"Dòng kẻ thực tế bị xiên {actual_skew:.1f}°."}

    # THÀNH CÔNG
    return aligned_img, None

In [ ]:
def natural_sort_key(s):
    return [int(text) if text.isdigit() else text.lower() for text in re.split('([0-9]+)', s)]

def batch_extract_and_align_student_exams(json_path, template_dir, input_main_folder, output_main_folder):
    
    os.makedirs(output_main_folder, exist_ok=True)
    fallback_json_path = os.path.join(output_main_folder, "warp_errors_log.json")
    
    error_logs = []
    
    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            all_rois_data = json.load(f)
        template_filenames = sorted(all_rois_data.keys(), key=natural_sort_key)
        rois_per_page = [all_rois_data[fn].get("rois", []) for fn in template_filenames]
    except Exception as e:
        print(f"❌ Lỗi đọc file JSON: {e}")
        return

    student_folders = sorted(os.listdir(input_main_folder), key=natural_sort_key)
    
    for student_name in student_folders:
        student_in_path = os.path.join(input_main_folder, student_name)
        if not os.path.isdir(student_in_path): continue
            
        print(f"\n👨‍🎓 Đang xử lý bài làm: {student_name}")
        student_out_path = os.path.join(output_main_folder, student_name)
        os.makedirs(student_out_path, exist_ok=True)
        
        image_files = sorted([f for f in os.listdir(student_in_path) if f.lower().endswith(('.jpg', '.png', '.jpeg'))], key=natural_sort_key)
        
        for page_idx, img_filename in enumerate(image_files):
            if page_idx >= len(rois_per_page): break

            ext = os.path.splitext(img_filename)[1]
            img_path = os.path.join(student_in_path, img_filename)
            
            # Xác định key trang hiện tại, dùng để tra PAGE_MAP
            page_key = f"Trang_{page_idx + 1:02d}"
            page_names = PAGE_MAP.get(page_key, [])
            
            student_img_original = cv2.imread(img_path)
            if student_img_original is None: continue
            
            template_path = os.path.join(template_dir, template_filenames[page_idx])
            template_img = cv2.imread(template_path)
            if template_img is None: continue

            returned_img, error_info = module_1_align_images(template_img, student_img_original.copy())
            current_page_rois = rois_per_page[page_idx]
            
            if error_info is not None:
                print(f"   ⏩ Lỗi [{error_info['error_type']}]: {error_info['reason']}")
                
                error_logs.append({
                    "student_folder": student_name,
                    "filename": img_filename,
                    "page_index": page_idx + 1,
                    "error_type": error_info["error_type"],
                    "reason": error_info["reason"]
                })
                
                orig_filename = f"ERROR_Trang_{page_idx + 1:02d}_Original{ext}"
                cv2.imwrite(os.path.join(student_out_path, orig_filename), student_img_original)
                
                error_debug_img = returned_img.copy() 
                
                if error_info["error_type"] in ["GEOMETRY_WARP_ERROR", "HOUGH_SKEW_ERROR"]:
                    for roi_idx, coords in enumerate(current_page_rois):
                        x1, y1, x2, y2 = [int(c) for c in coords]
                        y1, y2 = max(0, y1), min(error_debug_img.shape[0], y2)
                        x1, x2 = max(0, x1), min(error_debug_img.shape[1], x2)
                        cv2.rectangle(error_debug_img, (x1, y1), (x2, y2), (0, 165, 255), 3)
                        cv2.putText(error_debug_img, f"ROI {roi_idx+1} (ERR)", (x1, y1 - 10), 
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 165, 255), 2)
                else:
                    cv2.putText(error_debug_img, f"WARP FAILED: {error_info['error_type']}", 
                                (50, 150), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 0, 255), 4)
                    cv2.putText(error_debug_img, "Cannot warp. Missing math matrix.", 
                                (50, 220), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 2)
                                
                debug_filename = f"ERROR_Trang_{page_idx + 1:02d}_Debug{ext}"
                cv2.imwrite(os.path.join(student_out_path, debug_filename), error_debug_img)
                continue
            
            # ======================================================
            # KHI THÀNH CÔNG: CẮT ROI & VẼ BBOX ĐỎ (dùng PAGE_MAP theo trang)
            # ======================================================
            debug_img = returned_img.copy() 
            for roi_idx, coords in enumerate(current_page_rois):
                x1, y1, x2, y2 = [int(c) for c in coords]
                y1, y2 = max(0, y1), min(returned_img.shape[0], y2)
                x1, x2 = max(0, x1), min(returned_img.shape[1], x2)
                
                crop_img = returned_img[y1:y2, x1:x2]
                
                # Lấy tên câu hỏi theo đúng vị trí trong trang, nếu vượt quá số lượng khai báo thì đặt tên Extra
                if roi_idx < len(page_names):
                    logical_name = page_names[roi_idx]
                else:
                    logical_name = f"Extra_ROI_{roi_idx + 1:02d}"
                
                cv2.imwrite(os.path.join(student_out_path, f"{page_key}_{logical_name}{ext}"), crop_img)
                
                cv2.rectangle(debug_img, (x1, y1), (x2, y2), (0, 0, 255), 3)
                cv2.putText(debug_img, f"ROI {roi_idx+1}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
            
            cv2.imwrite(os.path.join(student_out_path, f"{page_key}_Full_Box{ext}"), debug_img)
            
    if error_logs:
        with open(fallback_json_path, 'w', encoding='utf-8') as f_err:
            json.dump(error_logs, f_err, ensure_ascii=False, indent=4)
        print(f"\n⚠️ TỔNG KẾT: Có {len(error_logs)} lỗi phát sinh. Chi tiết tại: {fallback_json_path}")
    else:
        print("\n🎉 TỔNG KẾT: Không có trang nào bị lỗi!")

In [ ]:
# =========================================================
# ĐƯỜNG DẪN THỰC THI MÃ ĐỀ 1
# =========================================================
json_file = '/kaggle/input/datasets/camtran3506/template-1-rois/Template_1_ROIs.json'

# MỚI: Thư mục chứa 9 trang Template sạch để máy so sánh
template_dir = '/kaggle/input/datasets/camtran3506/hki-2025-2026/HKI2025_2026/Made_1/Template_1'

input_dir = '/kaggle/input/datasets/anartt/dts-hki2526/output/Made_1/Bai_lam'
output_dir = '/kaggle/working/Output_Ma_de_1'

batch_extract_and_align_student_exams(json_file, template_dir, input_dir, output_dir)

In [ ]:
# =========================================================
# ĐƯỜNG DẪN THỰC THI MÃ ĐỀ 2
# =========================================================
json_file = '/kaggle/input/datasets/camtran3506/template-2-rois/Template_2_ROIs.json'

# MỚI: Thư mục chứa 9 trang Template sạch để máy so sánh
template_dir = '/kaggle/input/datasets/camtran3506/hki-2025-2026/HKI2025_2026/Made_2/Template_2'

input_dir = '/kaggle/input/datasets/anartt/dts-hki2526/output/Made_2/Bai_lam'
output_dir = '/kaggle/working/Output_Ma_de_2'

batch_extract_and_align_student_exams(json_file, template_dir, input_dir, output_dir)